ESERCIZIO

Tunning e Tracking Integrato

1) Implementa una funzione obbiettivo per Optuna che addestri un classificatore semplice (es. su dastaset MNIST)
2) All'interno del trial, usa mlflow.log_params per salvare il leraning rate scelto da Optuna
3) Registra la precisione finale con mlflow.log_metric
4) Avvia 10 trial e apri l'interfaccia di ML Flow per identificare graficamente il trial migliore

Suggerimento: ricorda di avvisare il server MLFlow con il comando mlflow ui, per veder i risultati

In [8]:
# -*- coding: utf-8 -*-
"""MNIST, PyTorch, Optuna e MLflow: esempio didattico commentato.

Puoi eseguire il file .py oppure incollarne il contenuto in una cella Jupyter.
Dipendenze (nel terminale con .venv attivo):
    python -m pip install torch torchvision mlflow optuna
Non reinstallare torch se hai gia una versione CUDA funzionante.

Fonti ufficiali:
https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pytorch.html
https://optuna.readthedocs.io/en/stable/reference/generated/optuna.pruners.MedianPruner.html
https://docs.pytorch.org/docs/stable/data.html
"""

from pathlib import Path
import random

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import mlflow
import mlflow.pytorch
import optuna


# -----------------------------------------------------------------------------
# 1. CONFIGURAZIONE: modifica questi valori, senza cercarli dentro le funzioni.
# -----------------------------------------------------------------------------
N_TRIALS = 10           # Per una prima verifica imposta 1.
EPOCHS = 5             # Ogni trial completo esegue cinque epoche.
BATCH_SIZE = 64        # Immagini elaborate prima di ogni aggiornamento dei pesi.
SEED = 42
EXPERIMENT_NAME = "MNIST_Classification_Optuna"

# In uno script usiamo la cartella del file; in Jupyter la directory del kernel.
# In Jupyter questa NON coincide necessariamente con la cartella del notebook.
# Per una destinazione fissa puoi sostituire con Path(r"F:\EPICODE\mio_progetto").
BASE_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd().resolve()
)
DATA_DIR = BASE_DIR / "data"
DB_PATH = BASE_DIR / "mlflow_mnist.db"
ARTIFACT_DIR = BASE_DIR / "mlflow_artifacts_mnist"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    """Riduce la variabilita casuale; non garantisce identita su ogni hardware."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Evitiamo la selezione dinamica degli algoritmi cuDNN basata su benchmark.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# -----------------------------------------------------------------------------
# 2. DATI: scarichiamo e dividiamo MNIST UNA SOLA VOLTA per lo studio.
# -----------------------------------------------------------------------------
def get_datasets():
    transform = transforms.Compose([
        # Da immagine a tensore [1, 28, 28], con pixel in [0, 1].
        transforms.ToTensor(),
        # Trasformazione (pixel - 0.5) / 0.5: valori nell'intervallo [-1, 1].
        transforms.Normalize((0.5,), (0.5,)),
    ])
    full_train = datasets.MNIST(
        root=str(DATA_DIR), train=True, download=True, transform=transform
    )
    # Un generatore dedicato rende riproducibile la suddivisione 50k / 10k.
    # Tutti i trial vedono le STESSE immagini di training e validazione.
    train_dataset, val_dataset = random_split(
        full_train,
        [50000, 10000],
        generator=torch.Generator().manual_seed(SEED),
    )
    # Il test ufficiale (train=False) non viene utilizzato per cercare parametri.
    # Va riservato alla valutazione finale, dopo aver scelto la configurazione.
    return train_dataset, val_dataset


def get_loaders(train_dataset, val_dataset):
    # Ricreiamo il generatore per ogni trial: stesso ordine casuale iniziale.
    # num_workers=0 semplifica l'esecuzione nei notebook Windows.
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    return train_loader, val_loader


# -----------------------------------------------------------------------------
# 3. RETE: e una rete fully connected, non una CNN.
# -----------------------------------------------------------------------------
class SimpleNet(nn.Module):
    def __init__(self, n_hidden, n_layers):
        super().__init__()
        layers = [nn.Flatten()]  # [batch, 1, 28, 28] -> [batch, 784]
        in_features = 28 * 28
        for _ in range(n_layers):
            layers.append(nn.Linear(in_features, n_hidden))
            layers.append(nn.ReLU())
            in_features = n_hidden
        # Dieci output: un punteggio (logit) per ogni cifra da 0 a 9.
        layers.append(nn.Linear(in_features, 10))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Niente Softmax: CrossEntropyLoss lavora direttamente con i logits.
        return self.model(x)


# -----------------------------------------------------------------------------
# 4. FUNZIONE OBIETTIVO: Optuna la esegue per ogni configurazione.
# -----------------------------------------------------------------------------
def objective(trial, train_dataset, val_dataset):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    n_hidden = trial.suggest_int("n_hidden", 32, 128)
    n_layers = trial.suggest_int("n_layers", 1, 3)

    # Stesso seed per rendere il confronto piu controllato.
    # Architetture diverse consumano numeri casuali in modo diverso:
    # questo NON rende identiche le inizializzazioni tra reti diverse.
    set_seed(SEED)
    train_loader, val_loader = get_loaders(train_dataset, val_dataset)
    model = SimpleNet(n_hidden, n_layers).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Una run figlia per trial, dentro la run padre creata in main().
    with mlflow.start_run(run_name=f"trial_{trial.number:03d}", nested=True) as run:
        trial.set_user_attr("mlflow_run_id", run.info.run_id)
        mlflow.log_params({
            **trial.params, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
            "seed": SEED, "optimizer": "Adam", "device": str(DEVICE),
            "train_size": len(train_dataset), "val_size": len(val_dataset),
        })
        mlflow.set_tags({
            "dataset": "MNIST", "split_seed": str(SEED),
            "trial_number": str(trial.number), "trial_state": "running",
        })

        for epoch in range(EPOCHS):
            model.train()  # Modalita addestramento.
            train_loss_sum = 0.0
            for images, labels in train_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()               # Azzera i gradienti.
                logits = model(images)              # Forward pass.
                loss = criterion(logits, labels)    # Errore del batch.
                loss.backward()                     # Calcola i gradienti.
                optimizer.step()                    # Aggiorna i pesi.
                # Pesiamo per la dimensione reale del batch, incluso l'ultimo.
                train_loss_sum += loss.item() * images.size(0)
            train_loss = train_loss_sum / len(train_dataset)

            model.eval()  # Modalita valutazione: non aggiorniamo i pesi.
            val_loss_sum = 0.0
            correct = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)
                    logits = model(images)
                    val_loss_sum += criterion(logits, labels).item() * images.size(0)
                    # argmax sceglie la cifra con il punteggio piu alto.
                    correct += (logits.argmax(dim=1) == labels).sum().item()
            val_loss = val_loss_sum / len(val_dataset)
            val_accuracy = correct / len(val_dataset)

            # Le curve delle tre metriche saranno visibili nella dashboard.
            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
            }, step=epoch)
            print(
                f"Trial {trial.number:02d} | Epoca {epoch + 1}/{EPOCHS} | "
                f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
                f"val_accuracy={val_accuracy:.2%}"
            )

            # Optuna riceve un risultato INTERMEDIO per decidere il pruning.
            trial.report(val_accuracy, step=epoch)
            # All'ultima epoca il costo di training e gia sostenuto:
            # conserviamo il risultato finale invece di potare la prova.
            if epoch < EPOCHS - 1 and trial.should_prune():
                mlflow.set_tag("trial_state", "pruned")
                # L'eccezione comunica a Optuna che la prova e stata potata.
                # MLflow puo mostrarla come FAILED: usa il tag per distinguerla.
                raise optuna.TrialPruned()

        # Salviamo il modello dell'ULTIMA epoca e restituiamo la sua accuracy.
        # Non implementiamo early stopping o selezione dell'epoca migliore.
        # pickle evita il requisito input_example del formato pt2.
        mlflow.pytorch.log_model(
            pytorch_model=model, name="model", serialization_format="pickle"
        )
        mlflow.set_tag("trial_state", "complete")
        return val_accuracy


# -----------------------------------------------------------------------------
# 5. AVVIO: database MLflow, studio Optuna e riepilogo.
# -----------------------------------------------------------------------------
def main():
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    tracking_uri = "sqlite:///" + DB_PATH.as_posix()
    mlflow.set_tracking_uri(tracking_uri)

    # Se l'esperimento esiste, manteniamo la sua destinazione artifact originale.
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        mlflow.create_experiment(
            EXPERIMENT_NAME, artifact_location=ARTIFACT_DIR.as_uri()
        )
    mlflow.set_experiment(EXPERIMENT_NAME)

    print("Directory:", BASE_DIR)
    print("Dispositivo:", DEVICE)
    print("Tracking URI:", tracking_uri)
    train_dataset, val_dataset = get_datasets()

    # SQLite conserva anche lo studio Optuna, separato dal database MLflow.
    # Ogni avvio crea un nuovo studio; quelli precedenti restano archiviati.
    optuna_uri = "sqlite:///" + (BASE_DIR / "optuna_mnist.db").as_posix()
    study = optuna.create_study(
        storage=optuna_uri,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5, n_warmup_steps=2
        ),
    )
    with mlflow.start_run(run_name="studio_mnist"):
        mlflow.set_tag("optuna_study_name", study.study_name)
        # lambda passa i dataset alla funzione senza ricrearli per ogni trial.
        # n_jobs=1: trial sequenziali, adatti anche a una sola GPU.
        study.optimize(
            lambda trial: objective(trial, train_dataset, val_dataset),
            n_trials=N_TRIALS, n_jobs=1,
        )
        mlflow.log_metric("best_val_accuracy", study.best_value)
        mlflow.log_params({
            f"best_{key}": value for key, value in study.best_params.items()
        })
        mlflow.set_tag("best_trial_number", str(study.best_trial.number))

    print("\nMiglior accuratezza DI VALIDAZIONE:", f"{study.best_value:.2%}")
    print("Migliori iperparametri:", study.best_params)
    print("Trial migliore:", study.best_trial.number)
    print("Run MLflow:", study.best_trial.user_attrs["mlflow_run_id"])
    print("\nNon e ancora una valutazione sul test ufficiale MNIST.")
    print("\nApri un terminale con .venv attivo e copia TUTTO il comando:")
    print(
        f'mlflow ui --backend-store-uri "{tracking_uri}" '
        '--host 127.0.0.1 --port 5001'
    )
    print("Poi apri http://127.0.0.1:5001 e seleziona:", EXPERIMENT_NAME)
    return study


if __name__ == "__main__":
    # Anche in una cella Jupyter __name__ e normalmente '__main__'.
    study = main()



Directory: F:\EPICODE\Epicode_Python_AI_MachineLearning\PYTHON\Modulo_6_Generative\6_Full_StackAIApplication
Dispositivo: cuda
Tracking URI: sqlite:///F:/EPICODE/Epicode_Python_AI_MachineLearning/PYTHON/Modulo_6_Generative/6_Full_StackAIApplication/mlflow_mnist.db


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.07MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 311kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.81MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.53MB/s]
[I 2026-10-31 20:52:06,701] A new study created in RDB with name: no-name-2d6718c1-bb5e-4a29-94f5-2e566a0b20a0


Trial 00 | Epoca 1/5 | train_loss=0.4705 | val_loss=0.2788 | val_accuracy=91.67%
Trial 00 | Epoca 2/5 | train_loss=0.2128 | val_loss=0.1881 | val_accuracy=94.31%
Trial 00 | Epoca 3/5 | train_loss=0.1565 | val_loss=0.1601 | val_accuracy=94.80%
Trial 00 | Epoca 4/5 | train_loss=0.1250 | val_loss=0.1413 | val_accuracy=95.72%
Trial 00 | Epoca 5/5 | train_loss=0.1071 | val_loss=0.1198 | val_accuracy=96.20%


[I 2026-10-31 20:54:01,813] Trial 0 finished with value: 0.962 and parameters: {'lr': 0.0005611516415334506, 'n_hidden': 124, 'n_layers': 3}. Best is trial 0 with value: 0.962.


Trial 01 | Epoca 1/5 | train_loss=0.4554 | val_loss=0.3275 | val_accuracy=90.42%
Trial 01 | Epoca 2/5 | train_loss=0.2816 | val_loss=0.2769 | val_accuracy=92.14%
Trial 01 | Epoca 3/5 | train_loss=0.2326 | val_loss=0.2293 | val_accuracy=93.12%
Trial 01 | Epoca 4/5 | train_loss=0.2035 | val_loss=0.2031 | val_accuracy=93.95%
Trial 01 | Epoca 5/5 | train_loss=0.1811 | val_loss=0.2060 | val_accuracy=93.89%


[I 2026-10-31 20:55:44,117] Trial 1 finished with value: 0.9389 and parameters: {'lr': 0.0015751320499779737, 'n_hidden': 47, 'n_layers': 1}. Best is trial 0 with value: 0.962.


... (13 outputs left out)

[I 2026-10-31 21:05:14,570] Trial 8 finished with value: 0.9609 and parameters: {'lr': 0.000816845589476017, 'n_hidden': 108, 'n_layers': 1}. Best is trial 0 with value: 0.962.


Trial 09 | Epoca 1/5 | train_loss=0.4324 | val_loss=0.2856 | val_accuracy=91.84%
Trial 09 | Epoca 2/5 | train_loss=0.2416 | val_loss=0.2235 | val_accuracy=93.62%
Trial 09 | Epoca 3/5 | train_loss=0.1882 | val_loss=0.1874 | val_accuracy=94.34%
Trial 09 | Epoca 4/5 | train_loss=0.1577 | val_loss=0.1646 | val_accuracy=95.11%
Trial 09 | Epoca 5/5 | train_loss=0.1365 | val_loss=0.1573 | val_accuracy=95.29%


[I 2026-10-31 21:06:48,742] Trial 9 finished with value: 0.9529 and parameters: {'lr': 0.0010677482709481358, 'n_hidden': 89, 'n_layers': 1}. Best is trial 0 with value: 0.962.



Miglior accuratezza DI VALIDAZIONE: 96.20%
Migliori iperparametri: {'lr': 0.0005611516415334506, 'n_hidden': 124, 'n_layers': 3}
Trial migliore: 0
Run MLflow: 2856883d6ae84e349ab3017045d548f1

Non e ancora una valutazione sul test ufficiale MNIST.

Apri un terminale con .venv attivo e copia TUTTO il comando:
mlflow ui --backend-store-uri "sqlite:///F:/EPICODE/Epicode_Python_AI_MachineLearning/PYTHON/Modulo_6_Generative/6_Full_StackAIApplication/mlflow_mnist.db" --host 127.0.0.1 --port 5001
Poi apri http://127.0.0.1:5001 e seleziona: MNIST_Classification_Optuna
